In [ ]:
!pip install -q bitsandbytes accelerate
# !pip install flash-attn --no-build-isolation

In [ ]:
import gc
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

NEW_TOKENS = 100
print("GPU:", torch.cuda.get_device_name())
print("compute capability:", torch.cuda.get_device_capability())

## Passo 1 — Carga em 4-bit (QLoRA)

In [ ]:
cfg_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

modelo = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=cfg_bnb,
    device_map="auto",
)

vram_modelo = torch.cuda.memory_allocated() / 1024**2
print("vram apos carga:", round(vram_modelo), "MB")

## Passo 2 — Contexto médico massivo

In [ ]:
trecho = """A cefaleia primaria do tipo tensional apresenta-se com dor bilateral em pressao,
intensidade leve a moderada e duracao de 30 minutos a 7 dias. O diagnostico diferencial
com migranea sem aura exige atencao a presenca de nausea, fonofobia e fotofobia, criterios
presentes na classificacao da IHS. O tratamento agudo inclui AINEs como ibuprofeno 400mg
ou naproxeno 500mg. Profilaxia com amitriptilina 25mg quando a frequencia excede 4
episodios mensais. A infeccao do trato urinario nao complicada acomete mulheres em idade
fertil com queixas de disuria e polaciuria. O agente prevalente ehh Escherichia coli em
80-85 porcento dos casos. Urocultura com contagem superior a 100.000 UFC/mL confirma o
diagnostico. Tratamento: fosfomicina 3g dose unica ou nitrofurantoina 100mg 6/6h por 5
dias. A sindrome coronariana aguda manifesta-se com dor toracica retroesternal irradiando
para membro superior esquerdo. ECG de 12 derivacoes em ate 10 minutos. Elevacao de ST em
duas derivacoes contiguas sugere IAMCSST. Troponina I acima do percentil 99 confirma
injuria miocardica. Conduta: AAS 200mg mastigado, clopidogrel 300mg, cateterismo em ate
90 minutos. O hipotireoidismo primario cursa com TSH elevado e T4 livre baixo. Sintomas:
fadiga, ganho ponderal, intolerancia ao frio, bradicardia. Reposicao com levotiroxina
1.6 mcg/kg/dia em jejum. O diabetes mellitus tipo 2 caracteriza-se por resistencia
insulinica. Criterios: glicemia jejum acima de 126 mg/dL ou HbA1c acima de 6.5 porcento.
Primeira linha: metformina 500mg titulada ate 2g/dia. Segunda linha: SGLT2 (dapagliflozina)
ou GLP-1 (semaglutida) conforme perfil cardiovascular e renal do paciente."""

contexto_medico = trecho * 20

max_ctx = modelo.config.max_position_embeddings - NEW_TOKENS
toks_texto = tok.encode(contexto_medico)
n_bruto = len(toks_texto)

# monta prompt com chat template, trunca texto pra caber
toks_texto = toks_texto[:max_ctx - 80]
texto_trunc = tok.decode(toks_texto, skip_special_tokens=True)

messages = [
    {"role": "system", "content": "Voce eh um medico."},
    {"role": "user", "content": f"Resuma o contexto clinico abaixo:\n{texto_trunc}"},
]

prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
n_ctx = input_ids.shape[1]

print(f"tokens do RAG bruto: {n_bruto}")
print(f"prompt final: {n_ctx} tokens (max modelo: {modelo.config.max_position_embeddings}, reserva {NEW_TOKENS} pra gerar)")

## Passo 3 — Geração SEM cache

In [ ]:
# warmup
with torch.no_grad():
    modelo.generate(input_ids, max_new_tokens=1, do_sample=False)
torch.cuda.empty_cache()

modelo.config.use_cache = False
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

t0 = time.time()
with torch.no_grad():
    out_sem = modelo.generate(input_ids, max_new_tokens=NEW_TOKENS, do_sample=False)
torch.cuda.synchronize()
t_sem = time.time() - t0
pico_sem = torch.cuda.max_memory_allocated() / 1024**2

gerado = tok.decode(out_sem[0][n_ctx:], skip_special_tokens=True)
print(f"sem cache: {t_sem:.1f}s, pico {pico_sem:.0f} MB")
print("resposta:", gerado[:300])

## Passo 4 — Geração COM cache

In [ ]:
modelo.config.use_cache = True

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

t0 = time.time()
with torch.no_grad():
    out_com = modelo.generate(input_ids, max_new_tokens=NEW_TOKENS, do_sample=False)
torch.cuda.synchronize()
t_com = time.time() - t0
pico_com = torch.cuda.max_memory_allocated() / 1024**2

gerado = tok.decode(out_com[0][n_ctx:], skip_special_tokens=True)
print(f"com cache: {t_com:.1f}s, pico {pico_com:.0f} MB")
print("resposta:", gerado[:300])